# Bronze Layer - Raw Data Ingestion

This notebook ingests raw customer data from Databricks sample datasets.
No transformations - just load raw data into bronze table.

In [0]:
# Get environment parameters from job parameters or dbutils widgets
try:
    catalog = dbutils.widgets.get("catalog")
    schema = dbutils.widgets.get("schema")
except Exception:
    # Fallback for local development
    catalog = "gitflow"
    schema = "gitflow_dev"

print(f"Target Catalog: {catalog}")
print(f"Target Schema: {schema}")

## Read Raw Data from Databricks Sample Dataset

In [0]:
# Read sample customer data
df_raw = spark.read.csv(
    "/databricks-datasets/retail-org/customers/", header=True, inferSchema=True
)

print(f"Records read: {df_raw.count()}")
df_raw.display()

## Write to Bronze Table (No Transformations)

In [0]:
from pyspark.sql.functions import current_timestamp

# Add metadata columns
df_bronze = df_raw.withColumn("ingestion_timestamp", current_timestamp())

# Write to bronze table
table_name = f"{catalog}.{schema}.bronze_customers"
df_bronze.write.mode("overwrite").saveAsTable(table_name)

print(f"✅ Bronze table created: {table_name}")
print(f"Records written: {df_bronze.count()}")

## Verify Bronze Table

In [0]:
spark.sql(f"SELECT * FROM {catalog}.{schema}.bronze_customers LIMIT 10").display()